In [27]:
import json
import numpy as np
import matplotlib.pyplot as plt

def plot_metrics_loglog_with_fit(json_filepath):
    # 1. Load the JSON data
    with open(json_filepath, 'r') as f:
        data = json.load(f)
    
    # 2. Extract and sort the training set sizes
    sizes = sorted([int(k) for k in data.keys()])
    str_sizes = [str(s) for s in sizes]
    
    x_data = np.array(sizes, dtype=float)
    
    # Extract metric values
    energy_rmse = np.array([data[s]["rmse_e_per_atom"]["rmse_e_per_atom"] for s in str_sizes], dtype=float)
    force_rmse = np.array([data[s]["rmse_f"]["rmse_f"] for s in str_sizes], dtype=float)
    stress_rmse = np.array([data[s]["rmse_stress"]["rmse_stress"] for s in str_sizes], dtype=float)
    
    # Helper function to compute power-law fit and generate log-log plots
    def create_loglog_fit_plot(x, y, label, title, filename, color):
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # --- Power-Law Fitting ---
        # y = a * x^b  -->  ln(y) = ln(a) + b * ln(x)
        log_x = np.log(x)
        log_y = np.log(y)
        
        # Fit a 1st-degree polynomial (straight line) in log-log space
        b, log_a = np.polyfit(log_x, log_y, 1)
        a = np.exp(log_a)
        
        # Generate dense X values for a smooth fit line on the log scale
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = a * (x_fit ** b)
        
        # --- Plotting ---
        # Plot actual data points
        ax.scatter(x, y, color=color, s=50, zorder=5, label='Model RMSE')
        ax.plot(x, y, color=color, linestyle=':', alpha=0.5) # Faint dotted line connecting points
        
        # Plot power-law fit line
        fit_label = f'Fit: $RMSE = {a:.4f} \cdot N^{{{b:.3f}}}$'
        ax.plot(x_fit, y_fit, color='black', linestyle='--', linewidth=1.5, label=fit_label)
        
        # Configure axes to log-log scale
        ax.set_xscale('log')
        ax.set_yscale('log')
        
        # Setup specific tick formatting and clean grids for log-log scales
        ax.grid(True, which="both", linestyle='--', alpha=0.5)
        
        # Clean labels and titles
        ax.set_xlabel(r'Training Set Size $N$')
        ax.set_ylabel(f'{label} ')
        
        # Display the legend containing the power-law equation
        ax.legend(loc="upper center", bbox_to_anchor = (0.5, 1.4), frameon=True)
        
        # Force matplotlib to use explicit values on the X-axis for clarity
        # ax.set_xticks(x)
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        
        # Save plot to PDF with tight layout to prevent any text clipping
        plt.tight_layout()
        fig.savefig(filename, format='pdf')
        plt.close(fig)
        print(f"Successfully saved {filename} with power-law fit slope: {b:.4f}")

    # 3. Generate the 3 separate log-log plots with fits
    create_loglog_fit_plot(
        x=x_data, 
        y=energy_rmse, 
        label=r'Energy RMSE [$\mathrm{eV/atom}$]', 
        title='Energy RMSE Learning Curve', 
        filename='MC_energy_rmse_loglog.pdf', 
        color='#1f77b4'
    )
    
    create_loglog_fit_plot(
        x=x_data, 
        y=force_rmse, 
        label=r'Force RMSE [$\mathrm{eV/\AA}$]', 
        title='Force RMSE Learning Curve', 
        filename='MC_force_rmse_loglog.pdf', 
        color='#ff7f0e'
    )
    
    create_loglog_fit_plot(
        x=x_data, 
        y=stress_rmse, 
        label=r'Stress RMSE [$\mathrm{eV}/\AA^3$]', 
        title='Stress RMSE Learning Curve', 
        filename='MC_stress_rmse_loglog.pdf', 
        color='#2ca02c'
    )

<>:44: SyntaxWarning: invalid escape sequence '\c'
<>:44: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_28958/2969702936.py:44: SyntaxWarning: invalid escape sequence '\c'
  fit_label = f'Fit: $RMSE = {a:.4f} \cdot N^{{{b:.3f}}}$'


In [28]:
plot_metrics_loglog_with_fit("mace_results_summary.json")

Successfully saved MC_energy_rmse_loglog.pdf with power-law fit slope: -0.2172
Successfully saved MC_force_rmse_loglog.pdf with power-law fit slope: -0.1234
Successfully saved MC_stress_rmse_loglog.pdf with power-law fit slope: -0.1865
